# Notebook 05: Custom Tools & Multi-Agent Systems

*Advanced / Optional — Complexity Ladder Level 3-4*

You've mastered the agent loop, memory, planning, and reflection. Now we connect agents to the outside world through standardized tools (MCP) and explore how multiple agents collaborate.

**Expected time:** ~60 minutes  
**Prerequisites:** All prior notebooks

### Multi-Agent Interaction Patterns

#### 1. Supervisor Routing
```mermaid
graph TD
    User([User Input]) --> Supervisor{Supervisor Agent}
    Supervisor -->|Delegate| SpecialistA[Specialist A: Code/Math]
    Supervisor -->|Delegate| SpecialistB[Specialist B: Search/RAG]
    SpecialistA -->|Response| Supervisor
    SpecialistB -->|Response| Supervisor
    Supervisor --> FinalAnswer([Final Answer])
```

#### 2. Agent Debate
```mermaid
graph LR
    Propose[Agent A: Proposes Answer] --> Debate{Debate & Critique}
    Debate --> Critique[Agent B: Challenges/Refines]
    Critique --> Propose
    Debate --> Consensus([Final Consensus Answer])
```


## What You'll Learn

1. When and why to build custom tools (vs. using libraries)
2. MCP (Model Context Protocol) — the "USB-C for AI"
3. Build an MCP server with resources, tools, and prompts
4. Connect an MCP server to an agent
5. Multi-agent supervisor routing: one agent delegates to specialists
6. Multi-agent debate pattern: agents challenge each other


## Setup

This notebook has two modes:
- **Full mode** (USE_MCP = True): Install `mcp` SDK, run a real server
- **Concept mode** (USE_MCP = False): Pure Python simulation, no extra deps

In [1]:
!pip install mcp==1.28.1 openai==1.68.2  # optional: only needed for MCP full mode

USE_MCP = True  # Set True if you can install the mcp SDK

from agent_helpers import ReactAgent, make_tool, mock_llm, safe_calc
import os
import getpass
import json
from pathlib import Path

print("MCP mode:", "ON (full)" if USE_MCP else "Concept (pure Python)")

# --- Optional .env file loading ---
env_path = Path.cwd() / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    print("Loaded .env file.")
else:
    print("No .env file found — you will be prompted for API keys.")

# --- Provider config (for supervisor/debate patterns with real LLM) ---
PROVIDER = "openrouter"  # Choose: openrouter, groq, openai

MOCK = {
    "openrouter": False,
    "groq":       True,
    "openai":     True,
}

KEY_ENV_VAR = {
    "openrouter": "OPENROUTER_API_KEY",
    "groq":       "GROQ_KEY",
    "openai":     "OPENAI_API_KEY",
}

CONFIG = {
    "openrouter": {
        "name": "OpenRouter",
        "model": "deepseek/deepseek-v4-flash",
        "base_url": "https://openrouter.ai/api/v1",
        "key_url": "https://openrouter.ai/keys",
    },
    "groq": {
        "name": "Groq",
        "model": "llama-3.3-70b-versatile",
        "base_url": "https://api.groq.com/openai/v1",
        "key_url": "https://console.groq.com/keys",
    },
    "openai": {
        "name": "OpenAI",
        "model": "gpt-4o-mini",
        "base_url": None,
        "key_url": "https://platform.openai.com/api-keys",
    },
}

cfg = CONFIG[PROVIDER]
use_mock = MOCK[PROVIDER]

if use_mock:
    print(f"[{cfg['name']}] Mock mode enabled. Supervisor/debate will use canned responses.")
    client = None
    def make_model(client, model_name=None):
        return mock_llm()
else:
    from openai import OpenAI
    api_key = os.environ.get(KEY_ENV_VAR[PROVIDER]) or getpass.getpass(f"Enter your {cfg['name']} API key: ")
    kwargs = {"api_key": api_key}
    if cfg["base_url"]:
        kwargs["base_url"] = cfg["base_url"]
    client = OpenAI(**kwargs)
    
    def make_model(client, model_name=None):
        if model_name is None:
            model_name = cfg["model"]
        def fn(messages):
            try:
                r = client.chat.completions.create(model=model_name, messages=messages, max_tokens=500)
                if not r.choices:
                    return "Error: empty response from API"
                content = r.choices[0].message.content
                return content if content is not None else ""
            except Exception as e:
                return f"API Error: {type(e).__name__}: {e}"
        return fn

    print(f"[{cfg['name']}] API key configured. Model: {cfg['model']}")


MCP mode: ON (full)
Loaded .env file.
[OpenRouter] API key configured. Model: deepseek/deepseek-v4-flash


## 1. When to Build Custom Tools

Not every problem needs a custom tool. Build one when:

| Use Case | Example | Use Library or Build Tool? |
|----------|---------|---------------------------|
| Public API exists | Weather, search, currency | Use library directly |
| Internal/private data | Company DB, proprietary API | **Build tool wrapper** |
| Custom computation | Domain-specific calculation | **Build tool** |
| Hardware control | Arduino, sensors, robotic arm | **Build tool** |
| No standard interface exists | Legacy system | **Build tool + MCP** |


## 2. MCP — Model Context Protocol

MCP is a standard for connecting AI applications to external systems. Think of it as "USB-C for AI" — one protocol that works across models and tools.

```mermaid
graph LR
    A[LLM Host] <--> B[MCP Server]
    B --> C[Resource: file]
    B --> D[Tool: calculator]
    B --> E[Prompt: summarize]
```

Three building blocks:
- **Resource**: Readable data (files, DB queries)
- **Tool**: Action the model can invoke (functions)
- **Prompt**: Reusable prompt template


### MCP Server in Python (Full Mode)

If `USE_MCP = True`, this code builds a running MCP server. It requires the `mcp` package.

In [2]:
if USE_MCP:
    # Use FastMCP from the mcp Python SDK (v1.28.1+)
    from mcp.server.fastmcp import FastMCP

    fastmcp_server = FastMCP("agentic-course-server")

    @fastmcp_server.resource(uri="file:///notes", name="course_notes")
    def read_notes() -> str:
        return "Agentic AI uses planning, memory, tools, and reflection."

    @fastmcp_server.tool(name="calculator", description="Evaluate math expressions")
    def calculator(expr: str) -> str:
        return safe_calc(expr)

    @fastmcp_server.prompt(name="summarize", description="Summarize a topic")
    def summarize(topic: str) -> str:
        return f"Please provide a concise summary of {topic}."

    print("MCP server defined.")
    print("  Resources: file:///notes")
    print("  Tools: calculator")
    print("  Prompts: summarize")
    print("\nTo run: fastmcp_server.run()")
else:
    print("Set USE_MCP = True at the top to see the real SDK code.")
    print("Below is the conceptual equivalent in pure Python.")

MCP server defined.
  Resources: file:///notes
  Tools: calculator
  Prompts: summarize

To run: fastmcp_server.run()


### MCP Server (Concept Mode — Pure Python)

This simulates the same pattern without the MCP SDK. The structure is the same: resources, tools, prompts.

In [3]:
class MCPServer:
    """Minimal MCP-like server. Same concepts, no SDK dependency."""
    def __init__(self, name):
        self.name = name
        self.resources = {}
        self.tools = {}
        self.prompts = {}

    def add_resource(self, uri, name, fn):
        self.resources[uri] = {"name": name, "fn": fn}

    def add_tool(self, name, description, fn):
        self.tools[name] = {"description": description, "fn": fn}

    def add_prompt(self, name, description, template_fn):
        self.prompts[name] = {"description": description, "fn": template_fn}

    def get_resource(self, uri):
        r = self.resources.get(uri)
        if r:
            return r["fn"]()
        return f"Resource not found: {uri}"

    def call_tool(self, name, **kwargs):
        t = self.tools.get(name)
        if t:
            return t["fn"](**kwargs)
        return f"Tool not found: {name}"

    def get_prompt(self, name, **kwargs):
        p = self.prompts.get(name)
        if p:
            return p["fn"](**kwargs)
        return f"Prompt not found: {name}"

    def list(self):
        print(f"Server: {self.name}")
        print(f"  Resources: {list(self.resources)}")
        print(f"  Tools: {list(self.tools)}")
        print(f"  Prompts: {list(self.prompts)}")

print("MCPServer class defined.")


MCPServer class defined.


### Build a Domain-Specific MCP Server

Let's build a server relevant to your PBL project. It combines data, actions, and reusable prompts.

In [4]:
server = MCPServer("my-project-server")

# Resource: domain knowledge
server.add_resource("file:///domain", "domain_knowledge",
    lambda: "Agentic AI systems use loops, memory, planning, and reflection. "
        "The Complexity Ladder ranges from Level 1 (simple tool use) to Level 4 (multi-agent).")

# Resource: project notes
def read_project_notes():
    return ("Project goal: Build an agent that automates literature review. "
        "Key features: search papers, extract findings, generate summary.")
server.add_resource("file:///project-notes", "project_notes", read_project_notes)

# Tool: search domain knowledge
def domain_search(query):
    db = {
        "agent loop": "Plan -> Act -> Observe -> Adapt cycle.",
        "memory": "Short-term (context) and long-term (RAG) storage.",
        "planning": "Task decomposition into sub-steps.",
        "reflection": "Self-critique and iterative improvement.",
    }
    return db.get(query.lower(), f"No entry for: {query}")
server.add_tool("search", "Search course concepts", domain_search)

# Tool: to-do manager
todos = []
def add_todo(item):
    todos.append(item)
    return f"Added: {item} (total: {len(todos)})"
def list_todos():
    return "\n".join(f"  {i+1}. {t}" for i, t in enumerate(todos)) if todos else "  (empty)"
server.add_tool("add_todo", "Add a to-do item", add_todo)
server.add_tool("list_todos", "List all to-do items", list_todos)

# Prompt: project reflection
def reflection_prompt(progress):
    return f"Given my progress: {progress}, what should I do next on my project?"
server.add_prompt("project_reflection", "Reflect on project progress", reflection_prompt)

server.list()


Server: my-project-server
  Resources: ['file:///domain', 'file:///project-notes']
  Tools: ['search', 'add_todo', 'list_todos']
  Prompts: ['project_reflection']


### Using the MCP Server from an Agent

The agent doesn't call the MCP server directly. Instead, each tool in the MCP server becomes a tool the agent can use.

In [5]:
from agent_helpers import ReactAgent, make_tool

# With real API
if not USE_MCP:
    mcp_tools = [
        make_tool("search", "Search domain concepts", lambda q: server.call_tool("search", query=q)),
        make_tool("add_todo", "Add a to-do item", lambda i: server.call_tool("add_todo", item=i)),
    ]
    mock = mock_llm("TOOL: search, args: {\"query\": \"agent loop\"}")
    agent = ReactAgent(model=mock, tools=mcp_tools)
    result = agent.run("Search for agent loop concepts")
    print("Agent with MCP tools:")
    print(result)
    print("\nMCP server tools are used through the same agent interface.")
else:
    # Extract the tools from our real FastMCP server defined in cell 7
    mcp_tools = [
        make_tool(t.name, t.description, t.fn) 
        for t in fastmcp_server._tool_manager.list_tools()
    ]
    model_fn = make_model(client)
    agent = ReactAgent(model=model_fn, tools=mcp_tools)
    result = agent.run("Evaluate the expression (15 * 3) + 42 / 2")
    print("Agent with MCP tools:")
    print(result)


[07/19/26 20:49:35] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205300;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205301;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Agent with MCP tools:
Let's evaluate the expression step by step:  

The expression is:  
\( (15 \times 3) + 42 / 2 \)  

**Step 1:** Perform the multiplication inside the parentheses first.  
\( 15 \times 3 = 45 \)  

Now the expression becomes:  
\( 45 + 42 / 2 \)  

**Step 2:** Division comes before addition (order of operations: PEMDAS).  
\( 42 / 2 = 21 \)  

**Step 3:** Add the results.  
\( 45 + 21 = 66 \)  

\[
\boxed{66}
\]


## 3. Multi-Agent: Supervisor Routing

One agent (the supervisor) receives a task, decides which specialist agent should handle it, and routes the work.

```mermaid
graph TD
    S[Supervisor] -->|research task| R[Research Agent]
    S -->|writing task| W[Writer Agent]
    S -->|quality check| Q[Quality Agent]
    R --> S
    W --> S
    Q --> S
    S -->|final| U[User]
```

In [6]:
class SpecialistAgent:
    """A simple agent with a specific role."""
    def __init__(self, name, role_desc, model_fn):
        self.name = name
        self.role = role_desc
        self.model = model_fn

    def handle(self, task):
        return self.model([{"role": "user", "content": (
            f"You are a {self.role}. Respond to: {task}"
        )}])

class Supervisor:
    """Routes tasks to specialist agents."""
    def __init__(self, model_fn):
        self.model = model_fn
        self.specialists = {}

    def register(self, name, agent):
        self.specialists[name] = agent

    def run(self, task):
        # Decide which specialist to use
        decision = self.model([{"role": "user", "content": (
            f"Available specialists: {list(self.specialists)}.\n"
            f"Which should handle: {task}\nReply with just the name."
        )}])

        chosen = decision.strip().lower()
        agent = self.specialists.get(chosen)

        if agent:
            print(f"Supervisor delegates to: {agent.name}")
            result = agent.handle(task)
            return self.model([{"role": "user", "content": (
                f"Task: {task}\nResult from {agent.name}: {result}\n"
                f"Format a final response for the user:"
            )}])
        else:
            return f"No specialist available for: {chosen}"

if not USE_MCP:
    model_fn = mock_llm()
else:
    model_fn = make_model(client)

research = SpecialistAgent("researcher", "research specialist who finds information", model_fn)
writer = SpecialistAgent("writer", "technical writer who explains clearly", model_fn)
qa = SpecialistAgent("qa", "quality assurance specialist who checks for errors", model_fn)

sup = Supervisor(model_fn)
sup.register("researcher", research)
sup.register("writer", writer)
sup.register("qa", qa)

print("Supervisor routing system ready.")
print("  Specialists: researcher, writer, qa")
print("  Task: the supervisor decides who handles each request.")


Supervisor routing system ready.
  Specialists: researcher, writer, qa
  Task: the supervisor decides who handles each request.


### Run the Supervisor


In [7]:
for task in ["Find latest research on agentic AI", "Write a summary", "Check this text for errors"]:
    print(f">>> {task}")
    result = sup.run(task)
    print(f"  Final: {result[:120]}...\n")


>>> Find latest research on agentic AI


[07/19/26 20:49:40] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205306;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205307;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Supervisor delegates to: researcher


[07/19/26 20:49:42] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205312;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205313;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[07/19/26 20:49:50] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205318;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205319;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

...

[07/19/26 20:50:24] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205330;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205331;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[07/19/26 20:50:36] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205336;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205337;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

  Final: Here is the formatted final response based on the writer's result:

---

## What is a Summary?

A summary is a **concise...

>>> Check this text for errors


[07/19/26 20:50:44] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205342;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205343;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

  Final: No specialist available for: qaqa...



## 4. Multi-Agent: Debate Pattern

Two agents take opposing positions on a question. A meta-agent judges the arguments and produces a balanced conclusion.

```mermaid
graph TD
    Q[Question] --> A[Agent 1: Pro]
    Q --> B[Agent 2: Con]
    A --> J[Judge]
    B --> J
    J --> C[Conclusion]
```

In [8]:
class Debate:
    """Two agents debate, a judge decides."""
    def __init__(self, model_fn):
        self.model = model_fn

    def run(self, question, position_a, position_b):
        print(f"Question: {question}\n")

        arg_a = self.model([{"role": "user", "content": (
            f"Argue FOR this position: {position_a}\n"
            f"In context of: {question}"
        )}])
        print(f"--- Agent A (FOR: {position_a}) ---")
        print(f"{arg_a[:150]}...\n")

        arg_b = self.model([{"role": "user", "content": (
            f"Argue AGAINST this position: {position_a}\n"
            f"In context of: {question}"
        )}])
        print(f"--- Agent B (AGAINST: {position_a}) ---")
        print(f"{arg_b[:150]}...\n")

        verdict = self.model([{"role": "user", "content": (
            f"Question: {question}\n"
            f"Argument FOR: {arg_a}\n"
            f"Argument AGAINST: {arg_b}\n"
            f"Provide a balanced conclusion that weighs both sides:"
        )}])
        print("=== JUDGE'S VERDICT ===")
        print(verdict)
        return verdict

print("Debate class defined.")


Debate class defined.


### Run a Debate


In [9]:
if not USE_MCP:
    model_fn = mock_llm()
else:
    model_fn = make_model(client)

debate = Debate(model_fn)
debate.run(
    "Should AI agents operate autonomously without human approval?",
    "Full autonomy increases efficiency and speed",
    "Human oversight ensures safety and accountability"
)


Question: Should AI agents operate autonomously without human approval?



[07/19/26 20:50:45] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205348;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205349;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

--- Agent A (FOR: Full autonomy increases efficiency and speed) ---
Here is an argument **FOR** the position that full autonomy for AI agents increases efficiency and speed, within the context of whether they should op...



[07/19/26 20:50:54] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205354;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205355;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

--- Agent B (AGAINST: Full autonomy increases efficiency and speed) ---
While it's true that removing human oversight from AI agents can theoretically speed up decision-making and reduce bottlenecks, arguing **against** th...



[07/19/26 20:51:01] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205360;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205361;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

=== JUDGE'S VERDICT ===
### Balanced Conclusion

The debate over whether AI agents should operate autonomously without human approval ultimately hinges on a trade-off between **raw speed and efficiency** versus **accuracy, safety, and resilience**. Both sides present compelling arguments that cannot be dismissed outright.

- **The FOR argument** correctly identifies that human approval introduces latency, cognitive overhead, and resource waste—especially in time-critical domains like high-frequency trading, cybersecurity, and real-time logistics. In these contexts, full autonomy is not just beneficial but necessary to achieve optimal performance. The gains in speed and system efficiency are real and measurable.

- **The AGAINST argument** counters that speed without reliability is a hollow victory. Errors, edge cases, cascading failures, and unintended consequences (e.g., discrimination, market crashes, safety hazards) can negate any time savings, often at a far greater cost. Human ove

'### Balanced Conclusion\n\nThe debate over whether AI agents should operate autonomously without human approval ultimately hinges on a trade-off between **raw speed and efficiency** versus **accuracy, safety, and resilience**. Both sides present compelling arguments that cannot be dismissed outright.\n\n- **The FOR argument** correctly identifies that human approval introduces latency, cognitive overhead, and resource waste—especially in time-critical domains like high-frequency trading, cybersecurity, and real-time logistics. In these contexts, full autonomy is not just beneficial but necessary to achieve optimal performance. The gains in speed and system efficiency are real and measurable.\n\n- **The AGAINST argument** counters that speed without reliability is a hollow victory. Errors, edge cases, cascading failures, and unintended consequences (e.g., discrimination, market crashes, safety hazards) can negate any time savings, often at a far greater cost. Human oversight provides a

## Combining MCP with Multi-Agent

In a real system, MCP servers provide the tools that specialist agents use. Each specialist connects to its own MCP server:

```mermaid
graph TD
    S[Supervisor] --> R[Research Agent]
    S --> W[Writer Agent]
    R --> M1[MCP: Search DB]
    W --> M2[MCP: Document Store]
```

This architecture scales to complex systems while keeping components modular.

## What Breaks?


### Break 1: MCP Transport Mismatch

MCP supports stdio (same process), SSE (server-sent events), and HTTP. If the host expects one transport and the server provides another, they can't communicate.

In [10]:
print("Transport mismatch scenarios:")
print("  Host uses SSE but server runs on stdio -> connection refused")
print("  Server runs on HTTP but host expects SSE -> protocol error")
print("  Port already in use -> bind error")
print("\nFix: Verify transport type in both host config and server config.")


Transport mismatch scenarios:
  Host uses SSE but server runs on stdio -> connection refused
  Server runs on HTTP but host expects SSE -> protocol error
  Port already in use -> bind error

Fix: Verify transport type in both host config and server config.


### Break 2: MCP Version Mismatch

The MCP protocol is evolving. A v0.1 server won't work with a v0.2 host.

In [11]:
print("Version mismatch:")
print("  Server SDK: 0.1.0, Host SDK: 0.2.0")
print("  -> Handshake failure during initialization")
print("\nFix: Pin versions in requirements.txt. Use compatible SDK versions.")


Version mismatch:
  Server SDK: 0.1.0, Host SDK: 0.2.0
  -> Handshake failure during initialization

Fix: Pin versions in requirements.txt. Use compatible SDK versions.


### Break 3: Supervisor Can't Decide

If the supervisor can't classify a task, it may route to the wrong specialist or fail to route at all.

In [12]:
def confused_supervisor(task, model_fn):
    """Demonstrates a supervisor that can't decide."""
    decision = model_fn([{"role": "user", "content": (
        f"Specialists: [researcher, writer, qa]. Which handles: {task}\n"
        f"If unsure, reply: UNKNOWN"
    )}])
    if "UNKNOWN" in decision:
        print(f"Supervisor confused by: {task}")
        print("Fix: Add more specific routing rules or a default specialist.")
    else:
        print(f"Routed to: {decision.strip()}")

if not USE_MCP:
    model_fn = mock_llm()
else:
    model_fn = make_model(client)

confused_supervisor("Do the thing with the stuff", model_fn)


[07/19/26 20:51:08] INFO     HTTP Request: POST https://openrouter.ai/api/v1/chat/completions       ]8;id=15205366;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=15205367;file:///home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Supervisor confused by: Do the thing with the stuff
Fix: Add more specific routing rules or a default specialist.


### Break 4: Debate Never Converges

If the two agents argue different interpretations without acknowledging the other side, the judge can't produce a balanced conclusion.

In [13]:
print("Debate convergence failure:")
print("  Agent A: \"AI should be fully autonomous because...\"")
print("  Agent B: \"AI should never be autonomous because...\"")
print("  Judge: Can't reconcile absolute positions")
print("\nFix: Instruct agents to acknowledge the other side. ")
print("  Use: \"Argue FOR this position, but also address the strongest counterargument.\"")


Debate convergence failure:
  Agent A: "AI should be fully autonomous because..."
  Agent B: "AI should never be autonomous because..."
  Judge: Can't reconcile absolute positions

Fix: Instruct agents to acknowledge the other side. 
  Use: "Argue FOR this position, but also address the strongest counterargument."


## Connection to Agentic AI

This notebook covers **Complexity Ladder Level 4**: Multiple cooperating agents or persistent state across long tasks.

**Marking Grid alignment:**
- **Solution**: MCP-based architecture shows professional tool design
- **Demo**: Show how your agent handles failures gracefully
- **Presentation**: Multi-agent patterns demonstrate deep understanding

> "If you can explain WHY you chose Level 2 over Level 4, that's better than building a broken Level 4."
> — Course mantra


### PBL Reflection


In [14]:
print("Think about YOUR project:")
print("  1. Do you need an MCP server? Or is a simple tool wrapper enough?")
print("  2. Would your project benefit from multiple specialists or is one agent enough?")
print("  3. The course mantra: does your Complexity Ladder choice match your team's skills?")
print("  4. What's your fallback if you aim for Level 4 but run out of time?")
print("\nThe best project is one that WORKS, at any level of the ladder.")


Think about YOUR project:
  1. Do you need an MCP server? Or is a simple tool wrapper enough?
  2. Would your project benefit from multiple specialists or is one agent enough?
  3. The course mantra: does your Complexity Ladder choice match your team's skills?
  4. What's your fallback if you aim for Level 4 but run out of time?

The best project is one that WORKS, at any level of the ladder.


## Exercises


### Bronze — Run the MCP Server

Run the MCPServer code above with different resources and tools relevant to your domain. List what's registered.

In [15]:
# Build a server for YOUR domain
my_server = MCPServer("my-project")
my_server.add_resource("file:///kb", "knowledge_base",
    lambda: "Domain-specific knowledge goes here.")
my_server.add_tool("my_tool", "Describe your domain tool",
    lambda q: f"Processed domain query: {q}")
my_server.list()


Server: my-project
  Resources: ['file:///kb']
  Tools: ['my_tool']
  Prompts: []


### Silver — Add a Custom Tool to the MCP Server

Add a domain-specific tool to the MCPServer. For example: a PubMed search tool (Alejandro), a risk calculator (Jost), a schedule planner (Stamatela), or a cognitive load analyzer (Melike).

In [16]:
# --- Your domain tool ---
def your_domain_tool(param):
    """Implement your domain-specific logic."""
    # Example: query a database, run a calculation, search an API
    return f"Domain result for: {param}"
# -------------------------

my_server.add_tool("domain_tool", "Description of your domain tool", your_domain_tool)
print("Custom tool added.")
print(my_server.call_tool("domain_tool", param="test input"))


Custom tool added.
Domain result for: test input


### Gold — Supervisor with Two MCP Servers

Build two MCP servers (e.g., one for research, one for writing). Create a supervisor that routes tasks to the appropriate server. This is a full Level 4 architecture.

In [17]:
# MCP Server 1: Research
research_server = MCPServer("research-server")
research_server.add_tool("search_papers", "Search academic papers",
    lambda q: f"Found papers about: {q}")
research_server.add_tool("summarize_paper", "Summarize a paper",
    lambda t: f"Summary of: {t}")

# MCP Server 2: Writing
writing_server = MCPServer("writing-server")
writing_server.add_tool("draft", "Draft text",
    lambda t: f"Draft: {t}")
writing_server.add_tool("format", "Format output",
    lambda t: f"Formatted:\n{t}")

# Create specialist agents
class SpecialistAgent:
    def __init__(self, name, server):
        self.name = name
        self.server = server

    def handle(self, task):
        tool_name = "search_papers"
        if "search" in task.lower() or "find" in task.lower():
            tool_name = "search_papers"
        else:
            tool_name = list(self.server.tools.keys())[0]
        kwargs = {"q": task} if "search" in task.lower() else {"t": task}
        return self.server.call_tool(tool_name, **kwargs)

researcher = SpecialistAgent("researcher", research_server)
writer = SpecialistAgent("writer", writing_server)

# Supervisor routes tasks
def supervisor_route(task, model_fn):
    decision = model_fn([{"role": "user", "content": (
        f"Task: {task}\nAvailable: researcher, writer\n"
        f"Reply with just the name."
    )}])
    chosen = decision.strip().lower()
    if chosen == "researcher":
        print("-> Research Server")
        return researcher.handle(task)
    elif chosen == "writer":
        print("-> Writing Server")
        return writer.handle(task)
    return f"No server for: {chosen}"

print("Two-server system defined.")
print("Supervisor routes: find papers -> research server, write -> writing server")


Two-server system defined.
Supervisor routes: find papers -> research server, write -> writing server


## Next Steps

This is the final notebook. You now have the complete toolkit:

| Notebook | Skill | Ladder Level |
|----------|-------|-------------|
| 00 | Environment + first API call | — |
| 01 | Agent loop | Level 1 |
| 02 | Memory & RAG | Level 1-2 |
| 03 | Planning | Level 2-3 |
| 04 | Reflection | Level 2-3 |
| 05 | Custom tools & Multi-agent | Level 3-4 |

> "A well-executed Level 1 project beats a broken Level 4 one. Choose your level, justify it, and build something that works."
